# 01 — Momentum breakout backtest

**Project:** channel breakout momentum on synthetic OHLCV.

## Problem
Trend persistence is one of the best-documented anomalies in asset returns.
A practical question: does a simple *N*-day high/low breakout earn risk-adjusted
returns after costs on a noisy price path?

## Method
1. Generate synthetic OHLCV with `quant_utils.data.make_ohlcv`.
2. Signal with `PriceBreakout` / `ATRBreakout` (Turtle-style channel).
3. Evaluate with the vectorised `run_backtest` engine (1-bar lag, slippage + commission).
4. Report Sharpe, drawdown, and trade count from `summary_table`.

## Modules
- `quant_utils.strategies.momentum`
- `quant_utils.backtest.engine`
- `quant_utils.data.sample`


In [ ]:
from quant_utils.data import make_ohlcv
from quant_utils.strategies import PriceBreakout, ATRBreakout
from quant_utils.backtest import run_backtest, summary_table

df = make_ohlcv(n=504, drift=0.0004, vol=0.012, seed=7)
print(df.tail(3))


In [ ]:
pb = PriceBreakout(period=20)
atr = ATRBreakout(period=20, atr_period=14)

sig_pb = pb.generate_signals(df)
sig_atr = atr.generate_signals(df)

res_pb = run_backtest(sig_pb, df["Close"], slippage=0.0005, commission=0.001)
res_atr = run_backtest(sig_atr, df["Close"], slippage=0.0005, commission=0.001)

print("PriceBreakout")
print(summary_table(res_pb))
print()
print("ATRBreakout")
print(summary_table(res_atr))


In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-friendly
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
res_pb.equity_curve.plot(ax=ax, label="PriceBreakout")
res_atr.equity_curve.plot(ax=ax, label="ATRBreakout")
ax.set_title("Equity curves (synthetic)")
ax.set_ylabel("Equity (start=1)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Short result
On this seeded synthetic path, both breakouts trade infrequently relative to a
daily signal; ATR filtering typically cuts trades and can improve or worsen Sharpe
depending on the realised trend/noise mix. Inspect the printed metrics above.

## Limitations
- Synthetic GBM-like paths understate crashes, gaps, and regime shifts.
- Channel breakouts are highly parameter-sensitive; in-sample period choice overfits.
- Costs are flat bps — real futures/equity microstructure is state-dependent.
- Not a live trading system; research/education only.
